## Part 1: Parsing a Sentence with CKY

Given the following grammar in CNF.
1. Write the code for CKY algorithm and verify if the **sentence** follows or can be parsed by this grammar or not.
2. Also print the final CKY chart/table.

In [ ]:
import nltk
from nltk.parse import ChartParser
from nltk.grammar import CFG

In [ ]:
cky_grammar = CFG.fromstring("""
    S -> NP VP
    NP -> Det N
    VP -> V NP
    Det -> 'the'
    N -> 'man' | 'dog'
    V -> 'saw'
""")

sentence = "the man saw the dog"

In [ ]:
def cky_parse(sentence, grammar, start="S"):
    words = sentence.split()
    n = len(words)

    # Separate the CNF grammar's productions into lexical rules (A -> 'word')
    # and binary rules (A -> B C), since CKY needs to look these up quickly.
    lexical_rules = {}   # word -> list of nonterminals that produce it
    binary_rules = []    # list of (lhs, rhs1, rhs2)

    for prod in grammar.productions():
        lhs = str(prod.lhs())
        rhs = prod.rhs()
        if len(rhs) == 1 and isinstance(rhs[0], str):
            lexical_rules.setdefault(rhs[0], []).append(lhs)
        elif len(rhs) == 2:
            binary_rules.append((lhs, str(rhs[0]), str(rhs[1])))
        else:
            raise ValueError(f"Grammar rule not in CNF: {prod}")

    # chart[i][j] holds the set of nonterminals that derive words[i:j]
    chart = [[set() for _ in range(n + 1)] for _ in range(n + 1)]

    # Base case: fill in single-word spans using the lexical rules
    for i, word in enumerate(words):
        chart[i][i + 1].update(lexical_rules.get(word, []))

    # Recursive case: build up larger spans from smaller ones
    for length in range(2, n + 1):           # span length
        for i in range(0, n - length + 1):   # span start
            j = i + length                   # span end
            for k in range(i + 1, j):        # split point
                for lhs, b, c in binary_rules:
                    if b in chart[i][k] and c in chart[k][j]:
                        chart[i][j].add(lhs)

    result = start in chart[0][n]
    return chart, result


def print_cky_table(chart, sentence):
    """Pretty-prints the CKY chart as a triangular table."""
    words = sentence.split()
    n = len(words)
    col_width = 16

    print("CKY Chart  (rows = span start i, cols = span end j)\n")
    header = "i\\j".ljust(6) + "".join(str(j).ljust(col_width) for j in range(1, n + 1))
    print(header)
    print("-" * len(header))
    for i in range(n):
        row = str(i).ljust(6)
        for j in range(1, n + 1):
            if j <= i:
                cell = ""
            else:
                cats = chart[i][j]
                cell = ", ".join(sorted(cats)) if cats else "-"
            row += cell.ljust(col_width)
        print(row)
    print()
    print("Word index legend: " + ", ".join(f"{i}:{w}" for i, w in enumerate(words)))


# Run CKY on the given sentence and grammar
chart, can_parse = cky_parse(sentence, cky_grammar)

print(f"Sentence: \"{sentence}\"")
print(f"Can be parsed by the grammar? {can_parse}\n")

print_cky_table(chart, sentence)


Sentence: "the man saw the dog"
Can be parsed by the grammar? True

CKY Chart  (rows = span start i, cols = span end j)

i\j   1               2               3               4               5               
--------------------------------------------------------------------------------------
0     Det             NP              -               -               S               
1                     N               -               -               -               
2                                     V               -               VP              
3                                                     Det             NP              
4                                                                     N               

Word index legend: 0:the, 1:man, 2:saw, 3:the, 4:dog


## Part 2: Comparision with NLTK's chart parser
After implementing your CKY algorithm, compare its output for the given sentence with the output of NLTK's built-in chart parser using the `cky_grammar`. This comparison will help you verify if your implementation is correct.

In [ ]:
# Parse the same sentence with NLTK's built-in ChartParser using cky_grammar
parser = ChartParser(cky_grammar)
words = sentence.split()
nltk_trees = list(parser.parse(words))

nltk_can_parse = len(nltk_trees) > 0

print(f"NLTK ChartParser — number of parse trees found: {len(nltk_trees)}")
for tree in nltk_trees:
    print(tree)

print()
print(f"My CKY algorithm result   : {can_parse}")
print(f"NLTK ChartParser result  : {nltk_can_parse}")
print(f"Do the two implementations agree? {can_parse == nltk_can_parse}")


NLTK ChartParser — number of parse trees found: 1
(S (NP (Det the) (N man)) (VP (V saw) (NP (Det the) (N dog))))

My CKY algorithm result   : True
NLTK ChartParser result  : True
Do the two implementations agree? True


### Conclusion

Both my hand-written CKY algorithm and NLTK's `ChartParser` agree on whether the sentence **"the man saw the dog"** can be generated by `cky_grammar`. The CKY chart above also shows exactly which spans of the sentence correspond to which grammar category (e.g. `the man` → `NP`, `saw the dog` → `VP`), which is the extra bookkeeping information a plain chart parser doesn't show as directly.